In [7]:
"""
Amazon E-Commerce Sales Data - Advanced Feature Engineering
===========================================================
Run this script directly with: python amazon_sales_feature_engineering.py

Automatically loads from preprocessed data or re-runs preprocessing
if the file is not found.
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────
INPUT_PATH  = "data/preprocessed_amazon_sales.csv"
OUTPUT_PATH = "data/feature_engineered.csv"
PLOT_DIR    = "outputs/feature_plots"

os.makedirs(PLOT_DIR,    exist_ok=True)
os.makedirs("data",      exist_ok=True)
os.makedirs("outputs",   exist_ok=True)


# ─────────────────────────────────────────────
# HELPER
# ─────────────────────────────────────────────
def save_fig(name):
    path = os.path.join(PLOT_DIR, f"{name}.png")
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved plot : {path}")


# ─────────────────────────────────────────────
# GENERATE SAMPLE DATA (fallback if no CSV found)
# ─────────────────────────────────────────────
def generate_preprocessed_sample(n=5000):
    """
    Creates a synthetic preprocessed DataFrame that mirrors what
    amazon_sales_preprocessing.py would produce, so this script
    can run standalone without the real data.
    """
    print("\n  No preprocessed file found — generating synthetic data …")
    np.random.seed(42)
    rng = pd.date_range("2022-04-01", "2022-06-30", freq="h")

    categories  = ["Set", "Kurta", "Western Dress", "Top", "Ethnic Dress",
                   "Blouse", "Bottom", "Saree", "Dupatta", "Sock"]
    sizes       = ["XS", "S", "M", "L", "XL", "XXL", "3XL", "Free"]
    statuses    = ["Shipped", "Cancelled", "Delivered", "Returned", "Pending"]
    states      = ["Maharashtra", "Karnataka", "Telangana", "Tamil Nadu",
                   "Uttar Pradesh", "Delhi", "Gujarat", "Rajasthan"]

    dates = np.random.choice(rng, n)
    qty   = np.random.choice([1, 2, 3, 4, 5], n, p=[0.60, 0.20, 0.10, 0.05, 0.05])
    amt   = np.round(np.random.lognormal(6.2, 0.7, n), 2)

    df = pd.DataFrame({
        "order_id"           : [f"405-{np.random.randint(1e6,9e6):.0f}" for _ in range(n)],
        "date"               : pd.to_datetime(dates),
        "status"             : np.random.choice(statuses, n),
        "fulfilment"         : np.random.choice(["Amazon","Merchant"], n, p=[0.70,0.30]),
        "sales_channel"      : np.random.choice(["Amazon.in","Non-Amazon"], n, p=[0.90,0.10]),
        "category"           : np.random.choice(categories, n),
        "size"               : np.random.choice(sizes, n),
        "qty"                : qty,
        "amount"             : amt,
        "ship_state"         : np.random.choice(states, n),
        "b2b"                : np.random.choice([True, False], n, p=[0.05, 0.95]),
        "promotion_ids"      : np.random.choice(["No Promotion","IN_Core_1","IN_Summer_2",""], n,
                                                  p=[0.60,0.15,0.15,0.10]),
        "revenue"            : amt * qty,
        "profit"             : amt * qty * 0.30,
        "day_of_week"        : pd.to_datetime(dates).dayofweek,
        "day_of_month"       : pd.to_datetime(dates).day,
        "month"              : pd.to_datetime(dates).month,
        "quarter"            : pd.to_datetime(dates).quarter,
        "year"               : pd.to_datetime(dates).year,
        "is_weekend"         : (pd.to_datetime(dates).dayofweek >= 5).astype(int),
        "has_promotion"      : np.random.choice([0, 1], n, p=[0.60, 0.40]),
        "fulfilled_by_amazon": np.random.choice([0, 1], n, p=[0.30, 0.70]),
        "b2b_flag"           : np.random.choice([0, 1], n, p=[0.95, 0.05]),
        "category_enc"       : np.random.randint(0, len(categories), n),
        "status_enc"         : np.random.randint(0, len(statuses), n),
        "fulfilment_enc"     : np.random.randint(0, 2, n),
    })

    df["price_bucket"] = pd.cut(
        df["amount"],
        bins=[0, 200, 500, 1000, float('inf')],
        labels=["Budget","Mid-Range","Premium","Luxury"]
    )

    df.to_csv(INPUT_PATH, index=False)
    print(f"  Synthetic data saved → {INPUT_PATH}  ({n:,} rows)\n")
    return df


# ─────────────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────────────
print("="*60)
print(" LOADING PREPROCESSED DATA")
print("="*60)

candidates = [
    INPUT_PATH,
    "data/preprocessed_amazon_sales.csv",
    "data/cleaned_amazon_sales.csv",
    "preprocessed_amazon_sales.csv",
]

df = None
for path in candidates:
    if os.path.exists(path):
        df = pd.read_csv(path, low_memory=False)
        print(f"  Loaded : {path}")
        print(f"  Shape  : {df.shape[0]:,} rows × {df.shape[1]} columns")
        break

if df is None:
    df = generate_preprocessed_sample()

# Ensure date column is parsed
date_cols = [c for c in df.columns if 'date' in c]
if date_cols:
    df[date_cols[0]] = pd.to_datetime(df[date_cols[0]], errors='coerce')


# ─────────────────────────────────────────────
# 2. ADVANCED FEATURE ENGINEERING
# ─────────────────────────────────────────────
print("\n" + "="*60)
print(" ADVANCED FEATURE ENGINEERING")
print("="*60)

# 2a. Qty-based features
if 'qty' in df.columns:
    df['is_bulk_order'] = (df['qty'] > 3).astype(int)
    df['qty_squared']   = df['qty'] ** 2
    print("  Created : is_bulk_order, qty_squared")

# 2b. Amount-based features
if 'amount' in df.columns:
    df['log_amount']     = np.log1p(df['amount'])
    df['amount_per_qty'] = df['amount'] / (df['qty'] + 1e-9) if 'qty' in df.columns else df['amount']
    print("  Created : log_amount, amount_per_qty")

# 2c. Revenue & profit log transforms
if 'revenue' in df.columns:
    df['log_revenue'] = np.log1p(df['revenue'])
if 'profit' in df.columns:
    df['log_profit'] = np.log1p(df['profit'])
    print("  Created : log_revenue, log_profit")

# 2d. Category-level revenue statistics
cat_col = next((c for c in df.columns if c in ['category', 'category_enc']), None)
if cat_col and 'revenue' in df.columns:
    cat_stats = (
        df.groupby(cat_col)['revenue']
          .agg(['mean', 'median', 'std'])
          .rename(columns={'mean':   'cat_revenue_mean',
                           'median': 'cat_revenue_median',
                           'std':    'cat_revenue_std'})
          .reset_index()
    )
    df = df.merge(cat_stats, on=cat_col, how='left')
    print("  Created : cat_revenue_mean, cat_revenue_median, cat_revenue_std")

# 2e. State-level revenue statistics
state_col = next((c for c in df.columns if 'state' in c and '_enc' not in c), None)
if state_col and 'revenue' in df.columns:
    state_stats = (
        df.groupby(state_col)['revenue']
          .agg(['mean', 'sum'])
          .rename(columns={'mean': 'state_revenue_mean',
                           'sum':  'state_revenue_sum'})
          .reset_index()
    )
    df = df.merge(state_stats, on=state_col, how='left')
    print("  Created : state_revenue_mean, state_revenue_sum")

# 2f. Size one-hot encoding
if 'size' in df.columns:
    size_dummies = pd.get_dummies(df['size'], prefix='size')
    df = pd.concat([df, size_dummies], axis=1)
    print(f"  One-hot : size → {list(size_dummies.columns)}")

# 2g. Season + season dummies
if 'month' in df.columns:
    season_map = {
        12: 'Winter', 1: 'Winter',  2: 'Winter',
        3:  'Spring', 4: 'Spring',  5: 'Spring',
        6:  'Summer', 7: 'Summer',  8: 'Summer',
        9:  'Autumn', 10: 'Autumn', 11: 'Autumn'
    }
    df['season'] = df['month'].map(season_map).fillna('Unknown')
    season_dummies = pd.get_dummies(df['season'], prefix='season')
    df = pd.concat([df, season_dummies], axis=1)
    print("  Created : season + season dummies")

# 2h. High-value order flag
if 'revenue' in df.columns:
    threshold = df['revenue'].quantile(0.90)
    df['is_high_value'] = (df['revenue'] >= threshold).astype(int)
    print(f"  Created : is_high_value (revenue ≥ {threshold:.0f}, top 10 %)")

# 2i. Weekend × promotion interaction
if 'is_weekend' in df.columns and 'has_promotion' in df.columns:
    df['weekend_promo'] = df['is_weekend'] * df['has_promotion']
    print("  Created : weekend_promo (interaction feature)")

print(f"\n  Shape after feature engineering: {df.shape[0]:,} × {df.shape[1]}")


# ─────────────────────────────────────────────
# 3. CORRELATION ANALYSIS
# ─────────────────────────────────────────────
print("\n" + "="*60)
print(" CORRELATION ANALYSIS")
print("="*60)

num_df     = df.select_dtypes(include='number')
target_col = 'profit' if 'profit' in num_df.columns else 'revenue'

corr_with_target = (
    num_df.corr()[target_col]
          .dropna()
          .abs()
          .sort_values(ascending=False)
)

print(f"\nTop 15 features correlated with '{target_col}':")
print(corr_with_target.head(15).to_string())

top_feats = corr_with_target.head(15).index.tolist()
if len(top_feats) > 1:
    plt.figure(figsize=(12, 9))
    sns.heatmap(
        num_df[top_feats].corr(),
        annot=True, fmt='.2f', cmap='coolwarm',
        linewidths=0.5, square=True, cbar_kws={'shrink': 0.8}
    )
    plt.title(f"Feature Correlation Heatmap (Top 15 vs {target_col})", fontsize=14)
    save_fig("correlation_heatmap")


# ─────────────────────────────────────────────
# 4. DISTRIBUTION PLOTS
# ─────────────────────────────────────────────
print("\n" + "="*60)
print(" DISTRIBUTION PLOTS")
print("="*60)

# 4a. Revenue & Profit distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, col, colour in zip(axes, ['revenue', 'profit'], ['steelblue', 'darkorange']):
    if col in df.columns:
        ax.hist(np.log1p(df[col].dropna()), bins=50,
                color=colour, edgecolor='white', alpha=0.85)
        ax.set_title(f'log(1 + {col}) Distribution', fontsize=13)
        ax.set_xlabel(f'log(1 + {col})')
        ax.set_ylabel('Frequency')
        ax.grid(axis='y', alpha=0.3)
plt.suptitle("Revenue & Profit Distributions (log-scaled)", fontsize=15)
save_fig("revenue_profit_distribution")

# 4b. Monthly revenue trend
if 'month' in df.columns and 'revenue' in df.columns:
    monthly = df.groupby('month')['revenue'].sum().reset_index()
    plt.figure(figsize=(10, 5))
    plt.bar(monthly['month'], monthly['revenue'] / 1e6,
            color='teal', edgecolor='white', alpha=0.85)
    plt.title("Monthly Revenue Trend (Millions INR)", fontsize=14)
    plt.xlabel("Month")
    plt.ylabel("Revenue (Millions INR)")
    plt.xticks(range(1, 13))
    plt.grid(axis='y', alpha=0.3)
    save_fig("monthly_revenue_trend")

# 4c. Top categories by revenue
if 'category' in df.columns and 'revenue' in df.columns:
    cat_rev = (
        df.groupby('category')['revenue']
          .sum()
          .sort_values(ascending=False)
          .head(10)
    )
    plt.figure(figsize=(11, 5))
    cat_rev.plot(kind='bar', color='slateblue', edgecolor='white', alpha=0.85)
    plt.title("Top 10 Categories by Revenue", fontsize=14)
    plt.xlabel("Category")
    plt.ylabel("Revenue (INR)")
    plt.xticks(rotation=45, ha='right')
    plt.grid(axis='y', alpha=0.3)
    save_fig("category_revenue")

# 4d. Season vs Revenue box plot
if 'season' in df.columns and 'revenue' in df.columns:
    plt.figure(figsize=(9, 5))
    season_order = ['Spring', 'Summer', 'Autumn', 'Winter']
    present = [s for s in season_order if s in df['season'].unique()]
    df_plot  = df[df['season'].isin(present)]
    df_plot.boxplot(column='revenue', by='season',
                    positions=[present.index(s) for s in present],
                    patch_artist=True)
    plt.title("Revenue by Season")
    plt.suptitle("")
    plt.xlabel("Season")
    plt.ylabel("Revenue (INR)")
    plt.grid(axis='y', alpha=0.3)
    save_fig("season_revenue_boxplot")

# 4e. Fulfillment type vs Revenue
ful_col = next((c for c in df.columns if 'fulfilment' in c and '_enc' not in c), None)
if ful_col and 'revenue' in df.columns:
    ful_rev = df.groupby(ful_col)['revenue'].mean().sort_values(ascending=False)
    plt.figure(figsize=(7, 4))
    ful_rev.plot(kind='bar', color=['steelblue', 'coral'], edgecolor='white', alpha=0.85)
    plt.title("Average Revenue by Fulfilment Type", fontsize=13)
    plt.xlabel("Fulfilment")
    plt.ylabel("Avg Revenue (INR)")
    plt.xticks(rotation=0)
    plt.grid(axis='y', alpha=0.3)
    save_fig("fulfilment_revenue")

# 4f. Price bucket distribution
if 'price_bucket' in df.columns:
    bucket_counts = df['price_bucket'].value_counts().sort_index()
    plt.figure(figsize=(7, 4))
    bucket_counts.plot(kind='bar', color='mediumseagreen', edgecolor='white', alpha=0.85)
    plt.title("Order Count by Price Bucket", fontsize=13)
    plt.xlabel("Price Bucket")
    plt.ylabel("Number of Orders")
    plt.xticks(rotation=0)
    plt.grid(axis='y', alpha=0.3)
    save_fig("price_bucket_distribution")


# ─────────────────────────────────────────────
# 5. SAVE FEATURE-ENGINEERED DATASET
# ─────────────────────────────────────────────
df.to_csv(OUTPUT_PATH, index=False)

print("\n" + "="*60)
print(" FEATURE ENGINEERING COMPLETE")
print("="*60)
print(f"  Output file : {OUTPUT_PATH}")
print(f"  Rows        : {df.shape[0]:,}")
print(f"  Columns     : {df.shape[1]}")
print(f"  Plots saved : {PLOT_DIR}/")
print("  New features : is_bulk_order, qty_squared, log_amount,")
print("                 amount_per_qty, log_revenue, log_profit,")
print("                 cat_revenue_*, state_revenue_*, size_*,")
print("                 season, season_*, is_high_value, weekend_promo")
print("="*60)

 LOADING PREPROCESSED DATA
  Loaded : data/preprocessed_amazon_sales.csv
  Shape  : 5,000 rows × 35 columns

 ADVANCED FEATURE ENGINEERING
  Created : is_bulk_order, qty_squared
  Created : log_amount, amount_per_qty
  Created : log_revenue, log_profit
  Created : cat_revenue_mean, cat_revenue_median, cat_revenue_std
  Created : state_revenue_mean, state_revenue_sum
  One-hot : size → ['size_3XL', 'size_4XL', 'size_5XL', 'size_6XL', 'size_Free', 'size_L', 'size_M', 'size_S', 'size_XL', 'size_XS', 'size_XXL']
  Created : season + season dummies
  Created : is_high_value (revenue ≥ 2286, top 10 %)
  Created : weekend_promo (interaction feature)

  Shape after feature engineering: 5,000 × 62

 CORRELATION ANALYSIS

Top 15 features correlated with 'profit':
profit              1.000000
revenue             1.000000
log_profit          0.801236
log_revenue         0.800189
is_high_value       0.745350
amount              0.707343
log_amount          0.600705
qty                 0.545783
qty_